# EDA — ASVspoof 2019 LA

Desbalance (~1:9 genuino:falso), duración, ataques, género/speaker y nivel de silencio
(posible fuga: silencios iniciales). Requiere `protocols/asv19_*.csv` y el audio en
`dataset_roots.asv19`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

ROOT = Path("..")
PROTO = ROOT / "protocols"
ASV19 = ROOT / "data" / "ASVspoof2019_LA"  # ajustar si usas otra ruta

splits = {s: pd.read_csv(PROTO / f"asv19_{s}.csv", keep_default_na=False)
          for s in ("train", "dev", "eval")}
for name, df in splits.items():
    print(name, len(df), "label counts:", df.label.value_counts().to_dict())

In [ ]:
train = splits["train"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
train.label.map({0: "genuina", 1: "sintética"}).value_counts().plot.bar(ax=ax[0], title="Train: clases")
train.loc[train.label == 1, "attack"].value_counts().sort_index().plot.bar(ax=ax[1], title="Train: ataques")
fig.tight_layout()
fig.savefig(ROOT / "results" / "figures" / "eda_train_balance.png", dpi=120)
plt.show()

In [ ]:
# Muestra de duraciones y energía del primer segundo (posible fuga por silencio).
sample = train.sample(n=min(200, len(train)), random_state=0)
rows = []
for _, r in sample.iterrows():
    path = ASV19 / r.path
    if not path.is_file():
        continue
    wav, sr = sf.read(path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    first = wav[:sr]
    rows.append({
        "label": int(r.label),
        "dur_s": len(wav) / sr,
        "rms_1s": float(np.sqrt(np.mean(first ** 2))) if len(first) else 0.0,
    })
eda = pd.DataFrame(rows)
display(eda.groupby("label")[["dur_s", "rms_1s"]].describe())
if not eda.empty:
    eda.boxplot(column="rms_1s", by="label")
    plt.suptitle("")
    plt.title("RMS del 1er segundo por clase")
    plt.savefig(ROOT / "results" / "figures" / "eda_silence_leak.png", dpi=120)
    plt.show()